# Урок 1.3 — MDP на RoomBot

Цель: описать маленькую среду, найти политику через Value Iteration и проверить влияние наград и переходов.
Нужны только NumPy и Matplotlib из среды урока 1.2. В VS Code выберите ядро **Python (mlagents)**.
Выполняйте ячейки сверху вниз. Нейросеть и GPU здесь не нужны.


## 1. Модель и алгоритм

RoomBot движется по комнате 3×3. Батарея даёт +1, лужа −1; обе завершают эпизод.
Остальные шаги стоят −0.02, вероятность скольжения 0.1.
Состояние — координаты, действия — четыре направления. Размер таблиц зависит от GRID.

Политика задаёт действие; V — ожидаемый возврат из клетки, Q — возврат при конкретном первом действии.
Value Iteration обновляет V максимумом по действиям до малых изменений.
В evaluate_policy вместо максимума используется среднее по вероятностям действий.


In [ ]:
"""Урок 1.3: одна модель RoomBot и Value Iteration, без нейросети."""
import numpy as np
import matplotlib.pyplot as plt

GRID = 3
BATTERY = (GRID - 1, GRID - 1)
PUDDLES = {(1, 1)}
SLIP, STEP_REWARD, GAMMA = 0.1, -0.02, 0.95
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
NAMES = ["↑", "↓", "←", "→"]


def states():
    return [(r, c) for r in range(GRID) for c in range(GRID)]


def terminal(s):
    return s == BATTERY or s in PUDDLES


def reward(s):
    return 1.0 if s == BATTERY else -1.0 if s in PUDDLES else STEP_REWARD


def next_states(s, action):
    result = []
    for b, (dr, dc) in enumerate(MOVES):
        s2 = (min(max(s[0] + dr, 0), GRID - 1), min(max(s[1] + dc, 0), GRID - 1))
        result.append((1 - SLIP if b == action else SLIP / 3, s2))
    return result


def lookahead(V, s, action, gamma):
    return sum(p * (reward(s2) + gamma * V[s2]) for p, s2 in next_states(s, action))


def greedy_policy(V, gamma=GAMMA):
    return {s: max(range(4), key=lambda a: lookahead(V, s, a, gamma))
            for s in states() if not terminal(s)}


def value_iteration(gamma=GAMMA, tol=1e-9, max_iter=10000):
    V = dict.fromkeys(states(), 0.0)
    for iteration in range(1, max_iter + 1):
        new = {s: 0.0 if terminal(s) else max(lookahead(V, s, a, gamma) for a in range(4))
               for s in states()}
        delta = max(abs(new[s] - V[s]) for s in states())
        V = new
        if delta < tol:
            return V, greedy_policy(V, gamma), iteration
    raise RuntimeError("Лимит итераций: проверьте награды, gamma и достижимость терминалов")


def evaluate_policy(policy, gamma=GAMMA, tol=1e-9, max_iter=10000):
    """Политика задаётся вероятностями четырёх действий для каждой клетки."""
    V = dict.fromkeys(states(), 0.0)
    for _ in range(max_iter):
        new = {s: 0.0 if terminal(s) else sum(policy[s][a] * lookahead(V, s, a, gamma)
                                             for a in range(4)) for s in states()}
        if max(abs(new[s] - V[s]) for s in states()) < tol:
            return new
        V = new
    raise RuntimeError("Оценка политики не сошлась за max_iter")


def plot_values(V, policy, title):
    grid = np.array([[V[(r, c)] for c in range(GRID)] for r in range(GRID)])
    fig, ax = plt.subplots()
    image = ax.imshow(grid, cmap="viridis")
    for s in states():
        mark = "B" if s == BATTERY else "X" if s in PUDDLES else NAMES[policy[s]]
        ax.text(s[1], s[0], f"{V[s]:.2f}\n{mark}", ha="center", va="center", color="white")
    ax.set(title=title, xticks=range(GRID), yticks=range(GRID))
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    plt.show()




## 2. Запуск и проверка

Сравните V старта у случайной и найденной политики. Карта показывает ценности и выбранные действия;
B — батарея, X — лужа. Награда начисляется при входе в терминальную клетку, её будущая V равна нулю.


In [ ]:
V, policy, iterations = value_iteration()
random_policy = {s: [0.25] * 4 for s in states() if not terminal(s)}
random_V = evaluate_policy(random_policy)
print(f"Итераций: {iterations}; V*(0,0)={V[(0,0)]:.3f}")
print(f"Случайная политика: V(0,0)={random_V[(0,0)]:.3f}")
plot_values(V, policy, "RoomBot: ценность и выбранное действие")


## 3. Задания с параметрами

Меняйте параметры в первой ячейке кода и каждый раз выполняйте **все ячейки заново**.
1. Переставьте лужу: `PUDDLES = {(0, 1)}`. Сравните стрелки верхнего ряда.
2. Верните исходную комнату и задайте `SLIP = 0.5`. Сравните V старта и действия.
3. Задайте `STEP_REWARD = 0.0`, `GAMMA = 1.0`. Проверьте сходимость в этой комнате.
   При gamma=1 нет общей гарантии сжатия оператора, но конкретный пример может сходиться.
   Не требуется получать расходимость. При исходной комнате эта версия сходится.
4. Увеличьте `GRID = 5`, оставьте `BATTERY = (GRID - 1, GRID - 1)` и задайте,
   например, `PUDDLES = {(1, 1), (2, 3)}`. Координаты должны находиться внутри комнаты;
   батарея не должна совпадать с лужей. Сравните карту с исходной.

Сдайте параметры, V старта и карты со стрелками для каждого варианта. Значения разных сред курса
(RoomBot, FrozenLake и лабораторный GridWorld) напрямую не сравниваются.


## 4. Задание: сравнить с Policy Iteration

Верните исходные параметры и перезапустите предыдущие ячейки.
Алгоритм чередует **оценку текущей политики** и **выбор лучших действий**.
Для evaluate_policy каждую выбранную команду переводим в четыре вероятности: 1 для неё, 0 для остальных.
Ниже готовый короткий цикл. Сравните ценности и число внешних итераций с Value Iteration.
Учитывайте, что одна внешняя итерация PI содержит внутреннюю оценку политики;
меньше внешних итераций не обязательно означает меньше работы.


In [ ]:
policy_pi = {s: [0.25] * 4 for s in states() if not terminal(s)}
for iteration in range(1, 101):
    V_pi = evaluate_policy(policy_pi)
    actions_pi = greedy_policy(V_pi)
    improved = {s: np.eye(4)[a].tolist() for s, a in actions_pi.items()}
    if all(np.allclose(improved[s], policy_pi[s]) for s in improved):
        break
    policy_pi = improved
else:
    raise RuntimeError("Политика не стабилизировалась за 100 внешних итераций")
V_vi, actions_vi, vi_iterations = value_iteration()
print("Внешних итераций PI:", iteration, "итераций VI:", vi_iterations)
print("Максимальная разница V:", max(abs(V_pi[s] - V_vi[s]) for s in states()))
plot_values(V_pi, actions_pi, "Policy Iteration")


## Проверка результата

- Расчёт завершается, карта соответствует заданным батарее и лужам.
- Ценность найденной политики не ниже случайной с теми же параметрами.
- PI и VI дают близкие ценности; при равных оценках допустимы разные стрелки.
- Вы можете назвать состояние, действия, переходы, награды и дисконт.

Доказательства Беллмана и сходимости доступны в математическом хабе сайта.
Следующий урок 1.4: Q-learning учит оценки из опыта, без известной таблицы переходов.
